# Notebook 01 — Document Loading Tests

## Goal
This notebook tests whether our cybersecurity source documents can be loaded correctly from multiple formats before any cleaning, chunking, or vector indexing.

## File types to test
- PDF
- DOCX
- TXT

## Expected outcome
- Successful loading of all documents
- Basic metadata inspection
- Page-wise validation for PDFs
- Unified document list ready for next-stage preprocessing

# Standard library
from pathlib import Path
from typing import List, Dict
import os

# Data inspection
import pandas as pd

# LangChain document loaders
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import TextLoader

print("Imports loaded successfully.")

In [1]:
# Standard library
from pathlib import Path
from typing import List, Dict
import os

# Data inspection
import pandas as pd

# LangChain document loaders
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import TextLoader

print("Imports loaded successfully.")

Imports loaded successfully.


In [2]:
# ============================================================
# STEP 1 — Define project paths
# ============================================================

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("DATA_DIR exists:", DATA_DIR.exists())

PROJECT_ROOT: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag
DATA_DIR: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\raw
DATA_DIR exists: True


In [3]:
# ============================================================
# STEP 2 — List files present in data/raw
# ============================================================

all_files = sorted([p for p in DATA_DIR.iterdir() if p.is_file()])

print("Files found in data/raw:")
for f in all_files:
    print("-", f.name)

print("\nTotal files found:", len(all_files))

Files found in data/raw:
- audit.txt
- cui-ssp-template-final.docx
- NIST.CSWP.29.pdf
- nist.sp.800-61r2.pdf

Total files found: 4


In [4]:
# ============================================================
# STEP 3 — File type detector
# ============================================================

def detect_file_type(file_path: Path) -> str:
    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        return "pdf"
    elif suffix == ".docx":
        return "docx"
    elif suffix == ".txt":
        return "txt"
    else:
        return "unsupported"

In [5]:
# ============================================================
# STEP 4 — Generic loader function
# ============================================================

def load_single_document(file_path: Path):
    """
    Load one file using the appropriate LangChain loader.
    Returns a list of LangChain Document objects.
    """
    file_type = detect_file_type(file_path)

    if file_type == "pdf":
        loader = PyPDFLoader(str(file_path))
        docs = loader.load()

    elif file_type == "docx":
        loader = Docx2txtLoader(str(file_path))
        docs = loader.load()

    elif file_type == "txt":
        loader = TextLoader(str(file_path), encoding="utf-8")
        docs = loader.load()

    else:
        raise ValueError(f"Unsupported file type: {file_path.suffix}")

    return docs

In [6]:
# ============================================================
# STEP 5 — Load all supported documents
# ============================================================

loaded_documents = []
loading_summary = []

for file_path in all_files:
    file_type = detect_file_type(file_path)

    if file_type == "unsupported":
        print(f"Skipping unsupported file: {file_path.name}")
        continue

    try:
        docs = load_single_document(file_path)
        loaded_documents.extend(docs)

        loading_summary.append({
            "file_name": file_path.name,
            "file_type": file_type,
            "num_langchain_docs": len(docs),
            "status": "success"
        })

        print(f"Loaded successfully: {file_path.name} | type={file_type} | docs={len(docs)}")

    except Exception as e:
        loading_summary.append({
            "file_name": file_path.name,
            "file_type": file_type,
            "num_langchain_docs": 0,
            "status": f"failed: {str(e)}"
        })

        print(f"Failed to load: {file_path.name}")
        print("Error:", e)

print("\nTotal loaded LangChain document objects:", len(loaded_documents))

Loaded successfully: audit.txt | type=txt | docs=1
Loaded successfully: cui-ssp-template-final.docx | type=docx | docs=1
Loaded successfully: NIST.CSWP.29.pdf | type=pdf | docs=32
Loaded successfully: nist.sp.800-61r2.pdf | type=pdf | docs=80

Total loaded LangChain document objects: 114


In [7]:
# ============================================================
# STEP 6 — Show loading summary
# ============================================================

summary_df = pd.DataFrame(loading_summary)
display(summary_df)

,file_name,file_type,num_langchain_docs,status
0,audit.txt,txt,1,success
1,cui-ssp-template-final.docx,docx,1,success
2,NIST.CSWP.29.pdf,pdf,32,success
3,nist.sp.800-61r2.pdf,pdf,80,success


In [8]:
# ============================================================
# STEP 7 — Inspect metadata and text sample from each file
# ============================================================

for row in loading_summary:
    if row["status"] != "success":
        continue

    file_name = row["file_name"]
    matched_docs = [
        doc for doc in loaded_documents
        if Path(doc.metadata.get("source", "")).name == file_name
    ]

    print("=" * 100)
    print("FILE:", file_name)
    print("TYPE:", row["file_type"])
    print("NUMBER OF DOC OBJECTS:", len(matched_docs))

    if matched_docs:
        first_doc = matched_docs[0]
        print("METADATA:", first_doc.metadata)
        print("\nTEXT PREVIEW:\n")
        print(first_doc.page_content[:1200])
        print("\n")

FILE: audit.txt
TYPE: txt
NUMBER OF DOC OBJECTS: 1
METADATA: {'source': 'd:\\Abhishek\\Project\\LangChain\\cybersecurity-multi-doc-rag\\data\\raw\\audit.txt'}

TEXT PREVIEW:

AUDIT TRAILS

Audit trails maintain a record of system activity both by system and
application processes and by user activity of systems and applications.  In
conjunction with appropriate tools and procedures, audit trails can assist
in detecting security violations, performance problems, and flaws in
applications.  This bulletin focuses on audit trails as a technical control
and discusses the benefits and objectives of audit trails, the types of
audit trails, and some common implementation issues.

An audit trail is a series of records of computer events, about an
operating system, an application, or user activities.  A computer system
may have several audit trails, each devoted to a particular type of
activity.  Auditing is a review and analysis of management, operational,
and technical controls.  The auditor ca

In [9]:
# ============================================================
# STEP 8 — PDF page-level inspection
# ============================================================

pdf_docs = [
    doc for doc in loaded_documents
    if str(doc.metadata.get("source", "")).lower().endswith(".pdf")
]

print("Total PDF page documents:", len(pdf_docs))

for i, doc in enumerate(pdf_docs[:5]):
    print("=" * 100)
    print(f"PDF SAMPLE #{i+1}")
    print("SOURCE:", doc.metadata.get("source"))
    print("PAGE:", doc.metadata.get("page"))
    print("TEXT PREVIEW:\n")
    print(doc.page_content[:1000])
    print("\n")

Total PDF page documents: 112
PDF SAMPLE #1
SOURCE: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\raw\NIST.CSWP.29.pdf
PAGE: 0
TEXT PREVIEW:

National Institute of Standards and Technology 
This publication is available free of charge from: https://doi.org/10.6028/NIST.CSWP.29 
February 26, 2024  
The NIST Cybersecurity 
Framework (CSF) 2.0


PDF SAMPLE #2
SOURCE: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\raw\NIST.CSWP.29.pdf
PAGE: 1
TEXT PREVIEW:

T he NIST Cybersecurity Framework (CSF) 2.0 
 i 
NIST CSWP 29 
February 26, 2024 
Abstract
T
he NIST Cybersecurity Framework (CSF) 2.0 provides guidance to industry, government 
agencies, and other organizations to manage cybersecurity risks. It offers a taxonomy of high-
level cybersecurity outcomes that can be used by any organization — regardless of its size, 
sector, or maturity — to better understand, assess, prioritize, and communicate its 
cybersecurity efforts. The CSF does not prescribe how outc

In [10]:
# ============================================================
# STEP 9 — DOCX and TXT inspection
# ============================================================

non_pdf_docs = [
    doc for doc in loaded_documents
    if not str(doc.metadata.get("source", "")).lower().endswith(".pdf")
]

print("Total DOCX/TXT documents:", len(non_pdf_docs))

for i, doc in enumerate(non_pdf_docs[:5]):
    print("=" * 100)
    print(f"NON-PDF SAMPLE #{i+1}")
    print("SOURCE:", doc.metadata.get("source"))
    print("METADATA:", doc.metadata)
    print("TEXT PREVIEW:\n")
    print(doc.page_content[:1000])
    print("\n")

Total DOCX/TXT documents: 2
NON-PDF SAMPLE #1
SOURCE: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\raw\audit.txt
METADATA: {'source': 'd:\\Abhishek\\Project\\LangChain\\cybersecurity-multi-doc-rag\\data\\raw\\audit.txt'}
TEXT PREVIEW:

AUDIT TRAILS

Audit trails maintain a record of system activity both by system and
application processes and by user activity of systems and applications.  In
conjunction with appropriate tools and procedures, audit trails can assist
in detecting security violations, performance problems, and flaws in
applications.  This bulletin focuses on audit trails as a technical control
and discusses the benefits and objectives of audit trails, the types of
audit trails, and some common implementation issues.

An audit trail is a series of records of computer events, about an
operating system, an application, or user activities.  A computer system
may have several audit trails, each devoted to a particular type of
activity.  Auditing is a review a

In [11]:
# ============================================================
# STEP 10 — Basic statistics
# ============================================================

stats_rows = []

for doc in loaded_documents:
    source = doc.metadata.get("source", "unknown")
    file_name = Path(source).name if source != "unknown" else "unknown"
    file_type = Path(source).suffix.lower().replace(".", "") if source != "unknown" else "unknown"
    char_count = len(doc.page_content)
    word_count = len(doc.page_content.split())

    stats_rows.append({
        "file_name": file_name,
        "file_type": file_type,
        "page": doc.metadata.get("page", None),
        "char_count": char_count,
        "word_count": word_count
    })

stats_df = pd.DataFrame(stats_rows)
display(stats_df.head(20))

print("\nGrouped summary:")
display(
    stats_df.groupby(["file_name", "file_type"], dropna=False)
    .agg(
        num_docs=("file_name", "count"),
        total_chars=("char_count", "sum"),
        total_words=("word_count", "sum"),
        avg_chars=("char_count", "mean")
    )
    .reset_index()
)

,file_name,file_type,page,char_count,word_count
0,audit.txt,txt,NaN,20319,3105
1,cui-ssp-template-final.docx,docx,NaN,35629,4039
2,NIST.CSWP.29.pdf,pdf,0.0,200,24
3,NIST.CSWP.29.pdf,pdf,1.0,2284,316
4,NIST.CSWP.29.pdf,pdf,2.0,512,82
5,NIST.CSWP.29.pdf,pdf,3.0,2016,131
6,NIST.CSWP.29.pdf,pdf,4.0,3406,504
7,NIST.CSWP.29.pdf,pdf,5.0,2788,401
8,NIST.CSWP.29.pdf,pdf,6.0,2350,340
9,NIST.CSWP.29.pdf,pdf,7.0,2106,295



Grouped summary:


,file_name,file_type,num_docs,total_chars,total_words,avg_chars
0,NIST.CSWP.29.pdf,pdf,32,69590,9468,2174.6875
1,audit.txt,txt,1,20319,3105,20319.0000
2,cui-ssp-template-final.docx,docx,1,35629,4039,35629.0000
3,nist.sp.800-61r2.pdf,pdf,80,231248,31965,2890.6000


In [12]:
# ============================================================
# STEP 11 — Standardize metadata schema
# ============================================================

standardized_documents = []

for idx, doc in enumerate(loaded_documents):
    source_path = doc.metadata.get("source", "")
    source_path = str(source_path)

    file_path = Path(source_path) if source_path else None
    file_name = file_path.name if file_path else "unknown"
    doc_type = file_path.suffix.lower().replace(".", "") if file_path else "unknown"

    standardized_metadata = {
        "document_id": f"doc_{idx:04d}",
        "file_name": file_name,
        "source_path": source_path,
        "doc_type": doc_type,
        "page_number": doc.metadata.get("page", None),
        "loader_metadata": doc.metadata
    }

    doc.metadata = standardized_metadata
    standardized_documents.append(doc)

print("Standardized documents created:", len(standardized_documents))
print("\nSample standardized metadata:")
print(standardized_documents[0].metadata if standardized_documents else "No documents loaded")

Standardized documents created: 114

Sample standardized metadata:
{'document_id': 'doc_0000', 'file_name': 'audit.txt', 'source_path': 'd:\\Abhishek\\Project\\LangChain\\cybersecurity-multi-doc-rag\\data\\raw\\audit.txt', 'doc_type': 'txt', 'page_number': None, 'loader_metadata': {'source': 'd:\\Abhishek\\Project\\LangChain\\cybersecurity-multi-doc-rag\\data\\raw\\audit.txt'}}


In [13]:
# ============================================================
# STEP 12 — Final standardized inspection
# ============================================================

preview_rows = []

for doc in standardized_documents[:15]:
    preview_rows.append({
        "document_id": doc.metadata.get("document_id"),
        "file_name": doc.metadata.get("file_name"),
        "doc_type": doc.metadata.get("doc_type"),
        "page_number": doc.metadata.get("page_number"),
        "source_path": doc.metadata.get("source_path"),
        "text_preview": doc.page_content[:200].replace("\n", " ")
    })

preview_df = pd.DataFrame(preview_rows)
display(preview_df)

,document_id,file_name,doc_type,page_number,source_path,text_preview
0,doc_0000,audit.txt,txt,NaN,d:\Abhishek\Project\LangChain\cybersecurity-mu...,AUDIT TRAILS Audit trails maintain a record o...
1,doc_0001,cui-ssp-template-final.docx,docx,NaN,d:\Abhishek\Project\LangChain\cybersecurity-mu...,<<Insert name>> SYSTEM SECURITY PLAN \t ...
2,doc_0002,NIST.CSWP.29.pdf,pdf,0.0,d:\Abhishek\Project\LangChain\cybersecurity-mu...,National Institute of Standards and Technology...
3,doc_0003,NIST.CSWP.29.pdf,pdf,1.0,d:\Abhishek\Project\LangChain\cybersecurity-mu...,T he NIST Cybersecurity Framework (CSF) 2.0 ...
4,doc_0004,NIST.CSWP.29.pdf,pdf,2.0,d:\Abhishek\Project\LangChain\cybersecurity-mu...,NIST CSWP 29 The NIST Cybersecurity Framework...
5,doc_0005,NIST.CSWP.29.pdf,pdf,3.0,d:\Abhishek\Project\LangChain\cybersecurity-mu...,NIST CSWP 29 The NIST Cybersecurity Framework...
6,doc_0006,NIST.CSWP.29.pdf,pdf,4.0,d:\Abhishek\Project\LangChain\cybersecurity-mu...,NIST CSWP 29 The NIST Cybersecurity Framework...
7,doc_0007,NIST.CSWP.29.pdf,pdf,5.0,d:\Abhishek\Project\LangChain\cybersecurity-mu...,NIST CSWP 29 The NIST Cybersecurity Framework...
8,doc_0008,NIST.CSWP.29.pdf,pdf,6.0,d:\Abhishek\Project\LangChain\cybersecurity-mu...,NIST CSWP 29 The NIST Cybersecurity Framework...
9,doc_0009,NIST.CSWP.29.pdf,pdf,7.0,d:\Abhishek\Project\LangChain\cybersecurity-mu...,NIST CSWP 29 The NIST Cybersecurity Framework...


In [14]:
# ============================================================
# STEP 13 — Save loading report for later notebooks
# ============================================================

OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "extracted_docs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

loading_report_path = OUTPUT_DIR / "document_loading_summary.csv"
preview_report_path = OUTPUT_DIR / "document_loading_preview.csv"

summary_df.to_csv(loading_report_path, index=False)
preview_df.to_csv(preview_report_path, index=False)

print("Saved:", loading_report_path)
print("Saved:", preview_report_path)

Saved: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\extracted_docs\document_loading_summary.csv
Saved: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\extracted_docs\document_loading_preview.csv


In [15]:
# ============================================================
# STEP 14 — Final conclusion
# ============================================================

print("Notebook 01 completed.")
print(f"Total supported source files found: {len([f for f in all_files if detect_file_type(f) != 'unsupported'])}")
print(f"Total loaded document objects: {len(standardized_documents)}")

successful_files = summary_df[summary_df["status"] == "success"]["file_name"].tolist()
failed_files = summary_df[summary_df["status"] != "success"]["file_name"].tolist()

print("\nSuccessful files:")
for f in successful_files:
    print("-", f)

print("\nFailed files:")
if failed_files:
    for f in failed_files:
        print("-", f)
else:
    print("None")

Notebook 01 completed.
Total supported source files found: 4
Total loaded document objects: 114

Successful files:
- audit.txt
- cui-ssp-template-final.docx
- NIST.CSWP.29.pdf
- nist.sp.800-61r2.pdf

Failed files:
None
